# Ejecución del paper (codigo demo-python)

Paper: Mehendran, Y., Tang, M., & Lu, Y. (2025). *"Enhancing Software
Vulnerability Detection Through Adaptive Test Input Generation Using
Genetic Algorithm."* arXiv:2508.05923.

## SUT: `sut.py`

Parser JSON con 4 ramas de decision. Programa objetivo sobre el que
el AG maximiza branch coverage (Sec. 3.3).

In [ ]:
TOTAL_BRANCHES = 4  # B_total, Ecuacion 1 (Sec. 3.3)
NOMBRES_RAMAS = ["B1", "B2", "B3", "B4"]


def run_sut(individuo: dict) -> list:
    """
    Ejecuta el SUT sobre un individuo y devuelve que ramas cubre:
        B1 -> tiene clave "nombre"
        B2 -> tiene clave "edad"
        B3 -> tiene clave "activo"
        B4 -> tiene clave "rol"
    """
    cubiertas = [False, False, False, False]

    if "nombre" in individuo:       # rama B1
        cubiertas[0] = True
    if "edad" in individuo:         # rama B2
        cubiertas[1] = True
    if "activo" in individuo:       # rama B3
        cubiertas[2] = True
    if "rol" in individuo:          # rama B4
        cubiertas[3] = True

    return cubiertas

## Poblacion inicial: `population.py`

Sec. 3.1 (Initial Input Generation). Cold-start: sin seeds previos,
gramatica JSON simplificada acotada al SUT.

In [ ]:
import random

# Genes posibles y su generador de valor (production rules simplificadas, Sec. 3.1)
CAMPOS_POSIBLES = {
    "nombre": lambda: random.choice(["Alice", "Bob", "Carla", "Diego"]),
    "edad":   lambda: random.randint(1, 99),
    "activo": lambda: random.choice([True, False]),
    "rol":    lambda: random.choice(["admin", "user", "guest"]),
}


def generar_individuo() -> dict:
    """Genera un individuo (JSON valido) con un subconjunto aleatorio de campos."""
    claves = list(CAMPOS_POSIBLES.keys())
    n_campos = random.randint(1, len(claves))
    claves_elegidas = random.sample(claves, n_campos)
    return {clave: CAMPOS_POSIBLES[clave]() for clave in claves_elegidas}


def generar_poblacion_inicial(tamanio: int) -> list:
    """El paper usa 100 individuos (Sec. 3.1); aqui es configurable."""
    return [generar_individuo() for _ in range(tamanio)]

## Fitness: `fitness.py`

Fitness = branch coverage (Sec. 3.3, Ecuacion 1):

```
Fitness(x) = B_exec(x) / B_total * 100
```

In [ ]:
def fitness(individuo: dict) -> float:
    cubiertas = run_sut(individuo)
    b_exec = sum(cubiertas)
    return (b_exec / TOTAL_BRANCHES) * 100


def ramas_cubiertas(individuo: dict) -> list:
    cubiertas = run_sut(individuo)
    return [nombre for nombre, c in zip(NOMBRES_RAMAS, cubiertas) if c]

## Seleccion: `selection.py`

Tournament selection (Sec. 3.4). k=3, valor tipico en la literatura de AGs.

In [ ]:
def seleccion_por_torneo(poblacion: list, k: int = 3) -> dict:
    """Toma k individuos al azar y devuelve el de mayor fitness del subconjunto."""
    k = min(k, len(poblacion))
    candidatos = random.sample(poblacion, k)
    return max(candidatos, key=fitness)

## Crossover: `crossover.py`

One-point crossover (Sec. 3.2.1, Fig. 2). El paper prefiere one-point
sobre two-point/uniform porque preserva mejor las relaciones
estructurales dentro del JSON.

In [ ]:
def cruce_un_punto(padre1: dict, padre2: dict) -> tuple:
    """Corta cada padre en un punto aleatorio de su lista de claves e intercambia."""
    claves1 = list(padre1.keys())
    claves2 = list(padre2.keys())

    if not claves1 or not claves2:
        return dict(padre1), dict(padre2)

    punto1 = random.randint(1, len(claves1))
    punto2 = random.randint(1, len(claves2))

    hijo1 = {clave: padre1[clave] for clave in claves1[:punto1]}
    hijo1.update({clave: padre2[clave] for clave in claves2[punto2:]})

    hijo2 = {clave: padre2[clave] for clave in claves2[:punto2]}
    hijo2.update({clave: padre1[clave] for clave in claves1[punto1:]})

    return hijo1, hijo2

## Mutacion: `mutation.py`

Reorder mutation (Sec. 3.2.2, Fig. 3). Reordena claves sin tocar
valores; nunca produce un JSON invalido.

In [ ]:
def mutacion_reorder(individuo: dict) -> dict:
    claves = list(individuo.keys())
    random.shuffle(claves)
    return {clave: individuo[clave] for clave in claves}

## Ciclo evolutivo: `main.py`

Orquesta el ciclo completo (Fig. 1, Sec. 3):

```
Poblacion inicial (3.1) -> ejecutar contra el SUT -> fitness (3.3) -> fitness suficiente? SI: stop / NO: seleccion (3.4) -> crossover (3.2.1) -> mutacion (3.2.2) -> nueva generacion
```

In [ ]:
TAMANIO_POBLACION = 6
GENERACIONES_MAXIMAS = 5
FITNESS_OBJETIVO = 100.0  # 100% de branch coverage (Sec. 3.3)


def imprimir_individuo(etiqueta: str, individuo: dict) -> None:
    f = fitness(individuo)
    ramas = ramas_cubiertas(individuo) or ["ninguna"]
    print(f"  {etiqueta}: {individuo}")
    print(f"      fitness = {f:.1f}%  |  ramas cubiertas = {', '.join(ramas)}")


def correr_ciclo_evolutivo(seed=None) -> None:
    if seed is not None:
        random.seed(seed)
        print(f"[semilla fijada en {seed} para reproducibilidad]\n")

    print("=" * 72)
    print("PASO 1 - Poblacion inicial (Seccion 3.1 del paper, cold-start)")
    print("=" * 72)
    poblacion = generar_poblacion_inicial(TAMANIO_POBLACION)
    for i, ind in enumerate(poblacion, start=1):
        imprimir_individuo(f"Individuo {i}", ind)

    for gen in range(1, GENERACIONES_MAXIMAS + 1):
        print("\n" + "=" * 72)
        print(f"GENERACION {gen}")
        print("=" * 72)

        mejor = max(poblacion, key=fitness)
        mejor_fitness = fitness(mejor)
        print(f"Mejor fitness de la generacion actual: {mejor_fitness:.1f}%")

        # rombo "Decision" de la Figura 1 del paper
        if mejor_fitness >= FITNESS_OBJETIVO:
            print("\nCriterio de parada alcanzado: 100% de branch coverage.")
            print(f"Individuo que logro cobertura total: {mejor}")
            return

        print("\nPASO 2 - Seleccion por torneo (Seccion 3.4)")
        padre1 = seleccion_por_torneo(poblacion)
        padre2 = seleccion_por_torneo(poblacion)
        imprimir_individuo("Padre 1", padre1)
        imprimir_individuo("Padre 2", padre2)

        print("\nPASO 3 - Crossover de un punto (Seccion 3.2.1, Figura 2)")
        hijo1, hijo2 = cruce_un_punto(padre1, padre2)
        imprimir_individuo("Hijo 1 (antes de mutar)", hijo1)
        imprimir_individuo("Hijo 2 (antes de mutar)", hijo2)

        print("\nPASO 4 - Mutacion por reordenamiento (Seccion 3.2.2, Figura 3)")
        hijo1 = mutacion_reorder(hijo1)
        hijo2 = mutacion_reorder(hijo2)
        imprimir_individuo("Hijo 1 (mutado)", hijo1)
        imprimir_individuo("Hijo 2 (mutado)", hijo2)

        print("\nPASO 5 - Nueva generacion (reemplazo con elitismo simple)")
        # Los 2 individuos de menor fitness se reemplazan por los hijos;
        # el resto sobrevive. Reemplazo no fijado (asi dse describe en el paper), adoptado
        # aqui para mantener el tamanio de poblacion fijo.
        poblacion_ordenada = sorted(poblacion, key=fitness, reverse=True)
        sobrevivientes = poblacion_ordenada[:-2]
        poblacion = sobrevivientes + [hijo1, hijo2]

        for i, ind in enumerate(poblacion, start=1):
            imprimir_individuo(f"Individuo {i}", ind)

    print("\nSe alcanzo el numero maximo de generaciones sin cobertura total.")
    mejor = max(poblacion, key=fitness)
    print(f"Mejor individuo final: {mejor}  (fitness = {fitness(mejor):.1f}%)")

### Corrida reproducible (`seed=32`)

In [ ]:
correr_ciclo_evolutivo(seed=32)

### Corrida con semilla aleatoria

Corremos varias veces sin `seed` para ver la limitacion que mencionamos (a veces converge en 75%, no en 100%, cuando la poblacion inicial pierde el gen `activo`).

In [ ]:
correr_ciclo_evolutivo()

## Verificacion rapida (equivalente a `test_ga.py`)

In [ ]:
assert run_sut({}) == [False, False, False, False]
assert run_sut({"nombre": "A", "edad": 1, "activo": True, "rol": "user"}) == [True, True, True, True]
assert fitness({}) == 0.0
assert fitness({"nombre": "A", "edad": 1, "activo": True, "rol": "user"}) == 100.0

individuo = {"nombre": "A", "edad": 1, "activo": True, "rol": "user"}
mutado = mutacion_reorder(individuo)
assert set(individuo.items()) == set(mutado.items())

padre1, padre2 = {"nombre": "A", "edad": 1}, {"activo": True, "rol": "user"}
hijo1, hijo2 = cruce_un_punto(padre1, padre2)
genes_padres = set(padre1) | set(padre2)
assert set(hijo1) <= genes_padres and set(hijo2) <= genes_padres

print("OK: coincide con las 12 pruebas de test_ga.py en el repo")